# 步驟 1：資料載入與處理

讀取從 FinMind 快取下來的財務、價量資料，並轉換為寬表（Wide-form DataFrame）。

**資料範圍**
- 時間：2019-01-02 ~ 2026-03（7 年，含 COVID 崩盤與 AI 牛市）
- 股票：台股上市 + 上櫃，約 2511 支
- 快取目錄：`finmind_cache/`

**前置條件**
- 已執行 `data_loaders/01_fetch_finmind_data.py` 下載资料
- 已執行 `data_loaders/02_fetch_fundamental_data.py`（選用）

In [ ]:
import os, sys, glob, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm

CACHE = '../finmind_cache'

# ── 收盤價寬表（已預先合併）
close = pd.read_pickle(os.path.join(CACHE, 'close_wide.pkl'))
close.index = pd.to_datetime(close.index)
close.index.name = 'date'
close.columns.name = 'stock_id'

print(f'close shape: {close.shape}')
print(f'date range : {close.index[0].date()} ~ {close.index[-1].date()}')
print(f'stocks     : {close.shape[1]}')

In [ ]:
# ── OHLCV：high / low / trading_money（逐檔讀取再合併）
high_frames, low_frames, money_frames = [], [], []
for f in tqdm(glob.glob(os.path.join(CACHE, 'price/*.pkl')), desc='OHLCV'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        for col, lst in [('max', high_frames), ('min', low_frames), ('Trading_money', money_frames)]:
            if col in df.columns:
                tmp = df[['date', 'stock_id', col]].copy()
                tmp[col] = pd.to_numeric(tmp[col], errors='coerce')
                lst.append(tmp)
    except Exception:
        pass

def _wide(frames, col):
    df = pd.concat(frames, ignore_index=True)
    w = df.pivot_table(index='date', columns='stock_id', values=col)
    w.index = pd.to_datetime(w.index)
    w.index.name = 'date'
    return w

high_wide  = _wide(high_frames,  'max')
low_wide   = _wide(low_frames,   'min')
money_wide = _wide(money_frames, 'Trading_money')

print(f'high shape : {high_wide.shape}')
print(f'money shape: {money_wide.shape}')

In [ ]:
# ── 月營收
rev_frames = []
for f in tqdm(glob.glob(os.path.join(CACHE, 'revenue/*.pkl')), desc='Revenue'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        rev_frames.append(df[['date', 'stock_id', 'revenue']])
    except Exception:
        pass

rev_wide = (
    pd.concat(rev_frames, ignore_index=True)
    .pivot_table(index='date', columns='stock_id', values='revenue')
)
rev_wide.index = pd.to_datetime(rev_wide.index)
rev_wide.index.name = 'date'

# ── 三大法人
fg_frames, tr_frames = [], []
for f in tqdm(glob.glob(os.path.join(CACHE, 'institution/*.pkl')), desc='Institution'):
    try:
        df = pd.read_pickle(f)
        df['date'] = pd.to_datetime(df['date'])
        df['net'] = df['buy'] - df['sell']
        fg_frames.append(df[df['name'] == 'Foreign_Investor'][['date', 'stock_id', 'net']])
        tr_frames.append(df[df['name'] == 'Investment_Trust'][['date', 'stock_id', 'net']])
    except Exception:
        pass

foreign_net = _wide(fg_frames, 'net')
trust_net   = _wide(tr_frames, 'net')

print(f'revenue shape    : {rev_wide.shape}')
print(f'foreign_net shape: {foreign_net.shape}')
print(f'trust_net shape  : {trust_net.shape}')

In [ ]:
# ── 流動性快速檢查（30日均量）
LIQUIDITY_MIN = 30_000_000  # 3000萬台幣
liquid_mask   = money_wide.rolling(30).mean().iloc[-1] >= LIQUIDITY_MIN
liquid_stocks = liquid_mask[liquid_mask].index.tolist()
print(f'流動性達標（>{LIQUIDITY_MIN/1e6:.0f}M）：{len(liquid_stocks)} / {money_wide.shape[1]} 支')